<a href="https://colab.research.google.com/github/Sharif2138/African-Language-Health-QA-Challenge/blob/main/real_exp_2_and_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Small Dataset Creation

Training on the full 29,751 samples on Free Colab takes hours per experiment
and risks session disconnects. To iterate quickly across experiments, we create
a stratified small dataset by sampling 200 examples per language from train and
100 per language from val.

This gives us:
- train_small: 1,600 samples (200 × 8 languages)
- val_small:   800 samples  (100 × 8 languages)

Every language is equally represented so no single language dominates training.
The full test set (2,618 samples) is always used as-is for submission.

Run this section once. All experiments 2 through 8 load from these files.

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("/content/drive/MyDrive/datasets/train_clean.csv")
val   = pd.read_csv("/content/drive/MyDrive/datasets/val_clean.csv")
test  = pd.read_csv("/content/drive/MyDrive/datasets/test_clean.csv")

print("Full train:", train.shape)
print("Full val  :", val.shape)
print("Full test :", test.shape)

In [ ]:
def make_small(df, n_per_lang, seed=42):
    """
    Sample n_per_lang rows from each language subset.
    Stratified so all 8 languages are equally represented.
    """
    parts = []
    for lang in df["subset"].unique():
        chunk = df[df["subset"] == lang]
        parts.append(chunk.sample(min(n_per_lang, len(chunk)), random_state=seed))
    return pd.concat(parts).reset_index(drop=True)


train_small = make_small(train, 200)
val_small   = make_small(val,   100)

print("train_small:", train_small.shape)
print("val_small  :", val_small.shape)
print()
print("train_small language distribution:")
print(train_small["subset"].value_counts())

In [ ]:
train_small.to_csv("/content/drive/MyDrive/datasets/train_small.csv", index=False)
val_small.to_csv(  "/content/drive/MyDrive/datasets/val_small.csv",   index=False)

print("Saved train_small and val_small to Drive")

# Experiment 2: mT5-small + LoRA — Generative Baseline

## Objective

Establish the first generative baseline by fine-tuning a multilingual
encoder-decoder model on the health QA task.

## Hypothesis

A generative model that has been pre-trained on 101 languages and fine-tuned
on our multilingual health QA data should produce better answers than
retrieval alone, even with limited training data.

## Why mT5-small

mT5-small was pre-trained on mC4, a multilingual web corpus covering 101
languages including Swahili and several West African languages close to Akan.
The small variant has 300M parameters — large enough to capture multilingual
health concepts, small enough to fine-tune on a Free Colab T4 GPU.

## Why LoRA

Full fine-tuning updates all 300M parameters which requires more GPU memory
and time than Free Colab allows. LoRA freezes the pretrained weights and
inserts small trainable adapter matrices into the attention layers. With
r=4 we train roughly 0.3% of parameters — the model learns the task
without overwriting its multilingual knowledge.

## Key Decisions

- MAX_INPUT_LENGTH = 96: covers the 95th percentile of question tokens (74)
  with a buffer for non-English scripts that tokenize into more subwords
- MAX_TARGET_LENGTH = 192: covers the 75th percentile of answer tokens (209)
  — a deliberate tradeoff between answer completeness and training speed
  on the small dataset
- 3 epochs with eval every epoch: gives a real learning curve
- gradient_accumulation_steps=2 with batch_size=8: effective batch of 16
  for more stable gradient updates without increasing memory
- generation_num_beams=1: greedy decoding during experiments — fast
  and sufficient for comparing experiments. Beam search is tested in Exp 9.

In [ ]:
!pip install -q torchao==0.16.0
!pip install -q transformers datasets evaluate accelerate peft sentencepiece rouge-score

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from peft import LoraConfig, TaskType, get_peft_model

## Load Data

In [ ]:
train_small = pd.read_csv("/content/drive/MyDrive/datasets/train_small.csv")
val_small   = pd.read_csv("/content/drive/MyDrive/datasets/val_small.csv")
test        = pd.read_csv("/content/drive/MyDrive/datasets/test_clean.csv")

train_ds = Dataset.from_pandas(train_small)
val_ds   = Dataset.from_pandas(val_small)
test_ds  = Dataset.from_pandas(test)

print("train_ds :", len(train_ds))
print("val_ds   :", len(val_ds))
print("test_ds  :", len(test_ds))

## Tokenization

The tokenizer converts raw text into token IDs that mT5 can process.

We define two functions:
- tokenize: for train and val — tokenizes both input and output, masks
  padding in labels with -100 so the loss function ignores padding tokens
- tokenize_test: for test — no output column exists, input only

In [ ]:
MODEL_NAME        = "google/mt5-small"
MAX_INPUT_LENGTH  = 96
MAX_TARGET_LENGTH = 192

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer)

In [ ]:
def tokenize(examples):
    inputs = tokenizer(
        examples["input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )
    targets = tokenizer(
        text_target=examples["output"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )
    inputs["labels"] = [
        [t if t != tokenizer.pad_token_id else -100 for t in ids]
        for ids in targets["input_ids"]
    ]
    return inputs


def tokenize_test(examples):
    return tokenizer(
        examples["input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )


tok_train = train_ds.map(tokenize,      batched=True, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(tokenize,        batched=True, remove_columns=val_ds.column_names)
tok_test  = test_ds.map(tokenize_test,  batched=True, remove_columns=test_ds.column_names)

print(tok_train)

In [ ]:
# Verify labels are correctly masked
sample = tok_train[0]
print("input_ids (first 10) :", sample["input_ids"][:10])
print("labels    (first 10) :", sample["labels"][:10])
print("non-padding label tokens:", sum(t != -100 for t in sample["labels"]))

## Model and LoRA

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.float()

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=4,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["q", "v"],
    bias="none"
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## Data Collator

Dynamic padding: each batch is padded to the longest sequence in that batch,
not to a global maximum. This reduces GPU memory usage and speeds up training.

In [ ]:
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

## Evaluation Metric

ROUGE-1 measures unigram overlap between generated and reference answers.
ROUGE-L measures longest common subsequence — it captures fluency better
than ROUGE-1 for longer answers.

Both are used by the competition. We use ROUGE-1 as the primary metric
for selecting the best checkpoint.

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # clip predictions to valid vocab range — prevents OverflowError
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {
        "ROUGE1": round(scores["rouge1"], 4),
        "ROUGEL": round(scores["rougeL"], 4)
    }

## Training Configuration

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/checkpoints/exp2",

    num_train_epochs=3,
    learning_rate=3e-4,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    weight_decay=0.01,
    max_grad_norm=1.0,

    fp16=False,
    bf16=False,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="ROUGE1",
    greater_is_better=True,

    generation_max_length=192,
    generation_num_beams=1,

    predict_with_generate=True,

    logging_strategy="steps",
    logging_steps=10,
    save_total_limit=1,
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

## Training

In [ ]:
trainer.train()

## Save Best Model

In [ ]:
trainer.save_model("/content/drive/MyDrive/models/exp2/best_model")
tokenizer.save_pretrained("/content/drive/MyDrive/models/exp2/best_model")
print("Model saved to Drive")

## Validation Evaluation

We run predictions on the full val_small set (800 samples) to get ROUGE
scores. Because we used load_best_model_at_end=True, the trainer
automatically uses the best checkpoint from training.

In [ ]:
val_preds = trainer.predict(tok_val)
print("Validation metrics:", val_preds.metrics)

In [ ]:
pred_text = tokenizer.batch_decode(
    val_preds.predictions,
    skip_special_tokens=True
)

ref_text = tokenizer.batch_decode(
    np.where(val_preds.label_ids != -100,
             val_preds.label_ids,
             tokenizer.pad_token_id),
    skip_special_tokens=True
)

for i in range(5):
    print("="*70)
    print("LANGUAGE :", val_small.iloc[i]["subset"])
    print("QUESTION :", val_small.iloc[i]["input"][:150])
    print("REFERENCE:", ref_text[i][:200])
    print("GENERATED:", pred_text[i][:200])
    print()

## Learning Curves

In [ ]:
history    = pd.DataFrame(trainer.state.log_history)
train_logs = history.dropna(subset=["loss"])
eval_logs  = history.dropna(subset=["eval_ROUGE1"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_logs["step"], train_logs["loss"], color="steelblue")
axes[0].set_title("Exp2 — Training Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(eval_logs["epoch"], eval_logs["eval_ROUGE1"],
             marker="o", label="ROUGE-1", color="steelblue")
axes[1].plot(eval_logs["epoch"], eval_logs["eval_ROUGEL"],
             marker="o", label="ROUGE-L", color="coral")
axes[1].set_title("Exp2 — Validation ROUGE per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "/content/drive/MyDrive/plots/exp2_curves.png",
    dpi=300, bbox_inches="tight"
)
plt.show()
print("Plot saved to Drive")

## Test Predictions and Submission

Generate predictions for all 2,618 test samples and create the submission
file. The full test set is always used — reducing it would produce an
incomplete submission that Zindi cannot score correctly.

In [ ]:
from torch.utils.data import DataLoader

model.eval()

test_loader = DataLoader(
    tok_test,
    batch_size=8,
    collate_fn=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8
    )
)

test_answers = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for batch in test_loader:
    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=192,
            num_beams=1
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    test_answers.extend(decoded)

print("Generated:", len(test_answers), "answers")
print("Sample:", test_answers[0][:200])

In [ ]:
submission = pd.DataFrame({
    "ID":         test["ID"],
    "TargetRLF1": test_answers,
    "TargetR1F1": test_answers,
    "TargetLLM":  test_answers
})

submission.to_csv(
    "/content/drive/MyDrive/submission_files/submission_exp2.csv",
    index=False
)

print("Submission saved — rows:", len(submission))
print(submission.head(3))

# Experiment 3: mT5-small + LoRA — Decoder Fix

## What Changed From Experiment 2

Experiment 2 produced degenerate outputs — every generated answer
started with `<extra_id_0>` followed by repetitive token loops.

The root cause: mT5 was pretrained using a span-corruption objective
where the decoder is trained to predict masked spans starting with
sentinel tokens like `<extra_id_0>`. When `decoder_start_token_id`
is not explicitly set, mT5 falls back to this pretraining behaviour
instead of generating fluent answers.

Two fixes applied in this experiment:
1. `model.config.decoder_start_token_id = tokenizer.pad_token_id`
   Forces the decoder to start from the correct token for seq2seq
   generation rather than the span-filling sentinel token.
2. `no_repeat_ngram_size=3`
   Prevents the repetition loops seen in Exp 2 outputs by blocking
   any 3-gram from appearing more than once in a generated sequence.

Everything else is identical to Experiment 2 so the comparison is clean.

## Hypothesis

Fixing the decoder start token will dramatically increase ROUGE scores
and produce coherent multilingual health answers for the first time.

## Model and LoRA

The decoder start token is fixed here, before LoRA is applied,
so the correct configuration is baked into the model from the start.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.float()

# Fix decoder start token — this is the critical change from Exp 2
# mT5 defaults to <extra_id_0> from pretraining which breaks generation
# Setting it to pad_token_id tells the decoder to start generating answers correctly
model.config.decoder_start_token_id          = tokenizer.pad_token_id
model.config.forced_bos_token_id             = None
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.decoder_start_token_id = tokenizer.pad_token_id
model.generation_config.forced_bos_token_id  = None

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=4,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["q", "v"],
    bias="none"
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## Data Collator

In [ ]:
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

## Evaluation Metric

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    preds  = np.clip(preds, 0, tokenizer.vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {
        "ROUGE1": round(scores["rouge1"], 4),
        "ROUGEL": round(scores["rougeL"], 4)
    }

## Training

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/checkpoints/exp3",

    num_train_epochs=3,
    learning_rate=3e-4,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    weight_decay=0.01,
    max_grad_norm=1.0,

    fp16=False,
    bf16=False,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="ROUGE1",
    greater_is_better=True,

    generation_max_length=192,
    generation_num_beams=1,

    predict_with_generate=True,

    logging_strategy="steps",
    logging_steps=10,
    save_total_limit=1,
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## Save Best Model

In [ ]:
trainer.save_model("/content/drive/MyDrive/models/exp3/best_model")
tokenizer.save_pretrained("/content/drive/MyDrive/models/exp3/best_model")
print("Model saved to Drive")

## Inspect Validation Predictions

We inspect 5 examples manually to verify the model is now generating
coherent answers rather than sentinel tokens or repetitive loops.

In [ ]:
import transformers
transformers.logging.set_verbosity_error()

from torch.utils.data import DataLoader

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

val_loader = DataLoader(
    tok_val,
    batch_size=8,
    collate_fn=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8
    )
)

pred_text = []
ref_text  = []

for batch in val_loader:
    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels         = batch["labels"]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=192,
            num_beams=1,
            decoder_start_token_id=tokenizer.pad_token_id,
            forced_bos_token_id=None,
            no_repeat_ngram_size=3
        )

    decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    pred_text.extend(decoded_preds)

    # decode labels
    labels = torch.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    ref_text.extend(decoded_labels)

print(f"Generated {len(pred_text)} predictions")

In [ ]:
for i in range(5):
    print("="*70)
    print("LANGUAGE :", val_small.iloc[i]["subset"])
    print("QUESTION :", val_small.iloc[i]["input"][:150])
    print("REFERENCE:", ref_text[i][:200])
    print("GENERATED:", pred_text[i][:200])
    print()

## Learning Curves

In [ ]:
history    = pd.DataFrame(trainer.state.log_history)
train_logs = history.dropna(subset=["loss"])
eval_logs  = history.dropna(subset=["eval_ROUGE1"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_logs["step"], train_logs["loss"], color="steelblue")
axes[0].set_title("Exp3 — Training Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(eval_logs["epoch"], eval_logs["eval_ROUGE1"],
             marker="o", label="ROUGE-1", color="steelblue")
axes[1].plot(eval_logs["epoch"], eval_logs["eval_ROUGEL"],
             marker="o", label="ROUGE-L", color="coral")
axes[1].set_title("Exp3 — Validation ROUGE per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "/content/drive/MyDrive/plots/exp3_curves.png",
    dpi=300, bbox_inches="tight"
)
plt.show()
print("Plot saved")